<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Solution: Match products between two shops

**[Match products between two shops](https://learning.nextia-ai.com/courses/record-matching/m09/match-products-between-two-shops/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 9, case study 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** match real products between two online shops from start to end: normalise titles, block, compare, classify, compare methods under one measure, and choose a design with its thresholds and its review band.

| | |
|---|---|
| **Time** | About 60 minutes in the notebook (the lesson page has the full story) |
| **Needs** | An internet connection for the setup cells (about 4 MB) |
| **You should know** | The course's steps, from [A supervised pair classifier](https://learning.nextia-ai.com/courses/record-matching/m06/a-supervised-pair-classifier/) and [Human review](https://learning.nextia-ai.com/courses/record-matching/m08/human-review/) |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, scikit-learn 1.9.1, LightGBM 4.7.0, Splink 5.0.0 and DuckDB 2.0.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m09/match-products-between-two-shops/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M09-L01-match-products-between-two-shops/match-products-between-two-shops-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, scikit-learn 1.9.1, lightgbm 4.7.0, splink 5.0.0, duckdb 2.0.0, pyarrow 26.0.1, and matplotlib. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 rapidfuzz==3.14.6 scikit-learn==1.9.1 lightgbm==4.7.0 splink==5.0.0 duckdb==2.0.0 pyarrow==26.0.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'rapidfuzz', 'scikit-learn', 'lightgbm', 'splink', 'duckdb', 'pyarrow']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It downloads two small zip files (about 1 MB together) from the benchmark's own page at the University of Leipzig, and the shipped embeddings and recorded answers (about 3 MB) from the course's labs repository, into a folder `product-matching`. It checks every file's checksum. It needs an internet connection and no account.

You should see `Ready:` and the folders. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `product-matching`, then run the cell again. If it happens again, the file on the source page has changed: the notebook stops, so that it never runs on other data than the lesson.

**Source and licence.** "Benchmark datasets for entity resolution" by the database group of Prof. Erhard Rahm, University of Leipzig (https://dbs.uni-leipzig.de/research/projects/benchmark-datasets-for-entity-resolution), under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Paper: H. Köpcke, A. Thor and E. Rahm, "Evaluation of entity resolution approaches on real-world match problems", PVLDB 3(1), 2010. The shipped embeddings and recorded answers are made from these titles by Nextia Learning; nothing else was changed.

In [ ]:
# Setup: download the benchmark from its source and the shipped files from labs, and check every checksum.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

SOURCES = {  # the zip files on the Leipzig benchmark page, and their SHA-256
    "Abt-Buy.zip": ("https://dbs.uni-leipzig.de/files/datasets/Abt-Buy.zip", "b9ff2937d97371b7a00b0a97033213bfbdee24e8f1a0718a41ea8364472a8ff8"),
    "Amazon-GoogleProducts.zip": ("https://dbs.uni-leipzig.de/files/datasets/Amazon-GoogleProducts.zip", "fcd72e223d51ce5b0d6b1d68034a438ba95cf2940431e209a57910b067cc4811"),
}
LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/M09-L01-match-products-between-two-shops/"
FILES = {
    "data/embeddings/abt_int8.npy": "e9c150d3e415496ee01adead8af64857f77da1406ae00a34773cd4602fa58a9c",
    "data/embeddings/abt_scale.npy": "a1e772ccf44fd03f673fdf9c0b212ac26d4320a4f1aa4923ce614f8794603cda",
    "data/embeddings/buy_int8.npy": "723c6ddbcb69c7662042c7a7b5755dcb6cda09a8237d3ed4f5a4c88351b4b67a",
    "data/embeddings/buy_scale.npy": "8d4ca7b4771cf48e26b31606eddc05d630f7694222899b27c70b6603da0a0e1e",
    "data/embeddings/amazon_int8.npy": "c9a348d6fbbeafd7143ae388fe5c89d664938cc89248764229f78da925860088",
    "data/embeddings/amazon_scale.npy": "886b1191b7a02b0a8cacd9988823b5ff4ba40b9ed11486b5658d8eda62731b18",
    "data/embeddings/google_int8.npy": "01f13f856849796aa402fc909a8d65810710de56f757a03249b3b46b2eddda67",
    "data/embeddings/google_scale.npy": "143d1e49bc0d69b94dfa2095f5b95439de18ca7ccc19dfcec8414e2f0654e1d6",
    "data/embeddings/abt_ids.csv": "300f25d6a709f4a0a9bdba484c1a44379c3f7ea9a92d91f5e1c61ce15b2de886",
    "data/embeddings/buy_ids.csv": "7bdb644a9e5ec5358096053d1399bfdbe740b67a2c97093cdb3ac0cedff5b24e",
    "data/embeddings/amazon_ids.csv": "05d3e62cdf3e902c6e086c0e49435a21dd76ca6c2620ef010fc77d14a9438607",
    "data/embeddings/google_ids.csv": "02a73fa2fe8d463a46fe3299208b7f55633eb47730d7f279a7d20184b3ea6aa9",
    "data/embeddings/model.json": "dd89731b7a08b85357481fdc7a3b8b645765985b7fd9531d3ae24879e69de8da",
    "data/recorded/llm_band.jsonl": "7445891025f7621d69eaa9863f8cd351586a8504ad055af1a440a6ad7b66e4cf"
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("product-matching").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, expected):
    path = WORK / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        shutil.copy(local, path) if local.exists() else download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder product-matching and run again.")
    return path

for name, (url, expected) in SOURCES.items():
    with zipfile.ZipFile(fetch("data/" + name, url, expected)) as z:
        z.extractall(WORK / "data" / name.removesuffix(".zip").lower())
for name, expected in FILES.items():
    fetch(name, LABS + name, expected)
os.chdir(WORK)
print("Ready:", sorted(p.name for p in Path("data").iterdir()))

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines:

- `load()` reads two shops and their true matches; `split_products()` puts every product in train, validation or test.
- `with_labels()` marks each pair as a true match or not; `pair_measures()` gives pair precision, recall and F1 on one split; `blocking_measures()` gives pair completeness and reduction ratio.
- The check helpers `expect()` and `expect_hash()`: they say if your answer is right, without showing the answer.

Run the cell. It prints nothing.

In [ ]:
# Helpers: load the shops, split, normalise, measure, and checks that do not show the answer.
import hashlib, json, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 39

def load(folder, a_file, b_file, map_file):
    """Shop A, shop B and the true matches (id_a, id_b). The files are Latin-1 text."""
    read = lambda f: pd.read_csv(f"data/{folder}/{f}", encoding="latin-1", dtype=str, keep_default_na=False)
    a, b, m = read(a_file), read(b_file), read(map_file)
    m.columns = ["id_a", "id_b"]
    if "title" in a:   # Amazon calls the name "title"
        a = a.rename(columns={"title": "name"})
    return a, b, m

def split_products(a, b, m, shares=(0.5, 0.25, 0.25), seed=SEED):
    """train / validation / test for every product, by group: the products joined by true matches stay together."""
    parent = {}
    def find(x):
        while parent.setdefault(x, x) != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for x in list("a" + a.id) + list("b" + b.id):
        find(x)
    for x, y in zip("a" + m.id_a, "b" + m.id_b):
        parent[find(x)] = find(y)
    groups = sorted({find(x) for x in parent})
    order = np.random.default_rng(seed).permutation(len(groups))
    cut1, cut2 = int(shares[0] * len(groups)), int((shares[0] + shares[1]) * len(groups))
    names = np.empty(len(groups), dtype=object)
    names[order[:cut1]], names[order[cut1:cut2]], names[order[cut2:]] = "train", "validation", "test"
    of_group = dict(zip(groups, names))
    return {x: of_group[find(x)] for x in parent}

def with_labels(pairs, m, split):
    """Adds `same` (a true match) and `split` (the split of the shop-A product) to a table of pairs."""
    true = set(zip(m.id_a, m.id_b))
    return pairs.assign(same=[(x, y) in true for x, y in zip(pairs.id_a, pairs.id_b)],
                        split=("a" + pairs.id_a).map(split).to_numpy())

def pair_measures(called, m, split, which="test"):
    """Pair precision, recall and F1 on one split: `called` is the pairs (id_a, id_b) a matcher calls the same."""
    true = {(x, y) for x, y in zip(m.id_a, m.id_b) if split["a" + x] == which}
    c = {(x, y) for x, y in zip(called.id_a, called.id_b) if split["a" + x] == which}
    right = len(c & true)
    p = right / len(c) if c else 0.0
    r = right / len(true) if true else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(f1, 3), "called": len(c), "right": right,
            "true": len(true)}

def blocking_measures(pairs, m, n_a, n_b):
    """Pair completeness (true matches kept) and reduction ratio (share of all pairs removed) of a blocking."""
    true = set(zip(m.id_a, m.id_b))
    kept = sum((x, y) in true for x, y in zip(pairs.id_a, pairs.id_b))
    return {"candidates": len(pairs), "true kept": kept, "pair completeness": round(kept / len(true), 4),
            "reduction ratio": round(1 - len(pairs) / (n_a * n_b), 5)}

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A small price-comparison website shows the offers of two online shops side by side, so that its readers can compare prices. The two shops here are **Abt** and **Buy**, two real online shops that sold electronics. The site's team must know which offer in one shop is the same product as an offer in the other shop.

Each error has a price. A **false match** shows two different products as one: a reader sees the wrong price and loses trust in the site. A **missed match** hides a comparison that the site could show. The team accepts matches automatically only when they are almost always right; a person checks the unsure ones.

You will build that matcher with the course's steps (normalise → block → compare → classify → measure), compare several methods under one measure, and choose a design with its thresholds and its review band.

## 2. The data

The benchmark comes from the database group of the University of Leipzig, under the CC BY 4.0 licence. The setup cell downloaded it from the source page. One row is one product offer of one shop. The third file lists the **true matches**: the pairs of offers that are the same product.

In [ ]:
abt, buy, matches = load("abt-buy", "Abt.csv", "Buy.csv", "abt_buy_perfectMapping.csv")
print("Abt:", abt.shape, abt.columns.tolist())
print("Buy:", buy.shape, buy.columns.tolist())
print("true matches:", len(matches))

> **Check.** Abt has 1,081 products and Buy 1,092, with 1,097 true matches. Abt has no `manufacturer` column.

Before any method, check the gaps and the repeats:

In [ ]:
print("repeated IDs:", int(abt.id.duplicated().sum()), int(buy.id.duplicated().sum()))
print("no price:", f"Abt {(abt.price == '').mean():.0%},", f"Buy {(buy.price == '').mean():.0%}")
print("no description in Buy:", f"{(buy.description == '').mean():.0%}")
print("products with a match:", f"Abt {abt.id.isin(matches.id_a).sum()} of {len(abt)},",
      f"Buy {buy.id.isin(matches.id_b).sum()} of {len(buy)}")
print("products with two or more matches:", f"Abt {(matches.id_a.value_counts() > 1).sum()},",
      f"Buy {(matches.id_b.value_counts() > 1).sum()}")

> **Check.** No repeated IDs. Many prices are missing (61% in Abt, 46% in Buy). **Every** product has a match in the other shop, and a few products have two. That is unusual: in real work, many products are in one shop only. Remember it when you read the results.

Five true matches, side by side:

In [ ]:
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200)
names_a, names_b = abt.set_index("id").name, buy.set_index("id").name
shown = matches.sample(5, random_state=4)
print(pd.DataFrame({"Abt": shown.id_a.map(names_a).to_numpy(), "Buy": shown.id_b.map(names_b).to_numpy()}).to_string(index=False))

The same product has two quite different titles. Abt ends each title with the model number. Buy writes the model number with hyphens, or not at all.

## 3. Split and baseline

The model will learn from labelled pairs, so you need a fair test. The split puts each product in **train**, **validation** or **test**, by group: products joined by true matches stay in one split. A pair belongs to the split of its Abt product. You choose every threshold on validation, and you look at test only to report.

In [ ]:
split = split_products(abt, buy, matches)
true_in = pd.Series([split["a" + x] for x in matches.id_a]).value_counts()
print("true matches per split:", true_in.to_dict())
print("all pairs:", f"{len(abt) * len(buy):,}")

The simplest matcher: the same name in both shops.

In [ ]:
same_name = abt.merge(buy, on="name", suffixes=("_a", "_b"))[["id_a", "id_b"]]
lower_name = (abt.assign(key=abt.name.str.lower().str.strip())
              .merge(buy.assign(key=buy.name.str.lower().str.strip()), on="key", suffixes=("_a", "_b"))[["id_a", "id_b"]])
print("exact name:", pair_measures(same_name, matches, split))
print("exact name, lower case:", pair_measures(lower_name, matches, split))

> **Check.** Exact names find one true match in the test split. Shops almost never write a title the same way.

## 4. Normalise titles and find model numbers

The `normalise()` function writes every title in one form: lower case, no signs, inch marks as `in`, no hyphens or slashes inside a word. Then `model_codes()` keeps the words that look like model numbers.

In [ ]:
UNITS = r"(in|gb|mb|tb|mp|w|hz|khz|mhz|ghz|mm|cm|ft|x|p|i|ms|v|mah|disc|port|pk|pack)"

def normalise(title):
    """Lower case, no accents or signs, inch marks as 'in', no hyphens or slashes inside a word."""
    s = "".join(c for c in unicodedata.normalize("NFKD", str(title)) if not unicodedata.combining(c)).lower()
    s = re.sub(r"[\u00ae\u2122\u00a9]", " ", s)                       # the signs (R), TM and (C)
    s = re.sub(r"(\d)\s*(?:'|\"|-inch\b|inch\b)", r"\1 in ", s)        # 42' and 42" -> 42 in
    s = re.sub(r"(?<=\w)[-/](?=\w)", "", s)                             # SD-V296 -> sdv296
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return " ".join(s.split())

def model_codes(title):
    """The model numbers in a normalised title: words of 4+ characters with letters and digits, not a size."""
    return {w for w in title.split() if len(w) >= 4 and re.search(r"\d", w) and re.search(r"[a-z]", w)
            and not re.fullmatch(r"\d+" + UNITS, w)}

for t in ["Toshiba Black DVD/VCR Combinaton Player - SDV296", "Toshiba SD-V296 DVD/VCR Combo",
          "Panasonic VIERA 42' Plasma Flat Panel HDTV In Black - TH42PX80U", "Panasonic Viera TH-42PX80U 42' Plasma TV"]:
    print(f"{t}\n   -> {normalise(t)}   codes {sorted(model_codes(normalise(t)))}")

In [ ]:
def prepare(df):
    """One row per product: the normalised title, its model codes and the price as a number."""
    title = df.name.map(normalise)
    price = pd.to_numeric(df.price.str.replace(r"[^0-9.]", "", regex=True), errors="coerce")
    text = (df.name + " " + df.get("description", "")).map(normalise)   # title and description, for codes
    return pd.DataFrame({"id": df.id, "name": df.name, "title": title, "codes": title.map(model_codes),
                         "price": price.where(price > 0), "text": text})

A, B = prepare(abt), prepare(buy)
print(A[["title", "codes", "price"]].head(3).to_string())
print("Abt products with a code:", f"{(A.codes.str.len() > 0).mean():.0%}", "| Buy:", f"{(B.codes.str.len() > 0).mean():.0%}")

> **Predict.** A rule says "same product when the two titles share a model code". Write down its precision and recall before you run the next cell.

In [ ]:
def code_pairs(A, B):
    """Every pair of products that share a model code."""
    ca, cb = A[["id", "codes"]].explode("codes").dropna(), B[["id", "codes"]].explode("codes").dropna()
    return ca.merge(cb, on="codes", suffixes=("_a", "_b"))[["id_a", "id_b"]].drop_duplicates()

same_code = code_pairs(A, B)
print("same model code:", pair_measures(same_code, matches, split))

> **Check.** Precision 0.954, recall 0.675. When the codes agree, the products are almost always the same. But a third of the true matches have no shared code.

### Failure case: a size that looks like a model number

The `model_codes()` function refuses words that are a number and a unit, such as `16gb` or `1080p`. Without that rule:

In [ ]:
loose = A.assign(codes=A.title.map(lambda t: {w for w in t.split() if len(w) >= 4 and re.search(r"\d", w) and re.search(r"[a-z]", w)}))
loose_b = B.assign(codes=B.title.map(lambda t: {w for w in t.split() if len(w) >= 4 and re.search(r"\d", w) and re.search(r"[a-z]", w)}))
loose_pairs = code_pairs(loose, loose_b)
print("same 'code' without the size rule:", pair_measures(loose_pairs, matches, split))
shared = loose.codes.explode().dropna()
print("the most common 'codes' in Abt:", shared.value_counts().head(4).to_dict())

> **Warning.** `1080p` is a screen resolution, shared by 28 Abt products. Without the size rule, every 1080p television "matches" every other 1080p product, and the precision falls to about one half. Check the most common values of any key before you trust it.

## 5. Blocking

All pairs of the two shops are 1,180,452. That is small enough for a laptop here, but not for a shop with a million products. **Blocking** keeps only candidate pairs. You use three blockings together:

- **same model code**;
- **TF-IDF top 5**: for each product, the 5 products of the other shop with the most similar title. TF-IDF compares the short pieces of letters that two titles share, and gives rare pieces more weight;
- **embeddings top 5**: the same with the cosine of the titles' embeddings (the shipped embeddings of the course's Granite model).

In [ ]:
def load_embeddings(shop, ids):
    """The shipped embeddings of one shop's titles (8-bit numbers times one scale per row), in the order of ids."""
    e = np.load(f"data/embeddings/{shop}_int8.npy").astype("float32") * np.load(f"data/embeddings/{shop}_scale.npy")
    row = {r: i for i, r in enumerate(pd.read_csv(f"data/embeddings/{shop}_ids.csv", dtype=str).id)}
    e = e[[row[x] for x in ids]]
    return e / np.linalg.norm(e, axis=1, keepdims=True)

from sklearn.feature_extraction.text import TfidfVectorizer

def similarity_tables(A, B, shop_a, shop_b):
    """Two tables of similarities, one row per product of A and one column per product of B: TF-IDF and cosine."""
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True).fit(pd.concat([A.title, B.title]))
    S_tfidf = (tfidf.transform(A.title) @ tfidf.transform(B.title).T).toarray()
    S_embed = load_embeddings(shop_a, A.id) @ load_embeddings(shop_b, B.id).T
    return S_tfidf, S_embed

S_tfidf, S_embed = similarity_tables(A, B, "abt", "buy")
print(S_tfidf.shape, "| TF-IDF of the first true match:", round(float(S_tfidf[A.id.eq(matches.id_a[0]).argmax(), B.id.eq(matches.id_b[0]).argmax()]), 3))

In [ ]:
def top_k(S, A, B, k):
    """For each product of A its k most similar products of B, and for each product of B its k most similar of A."""
    ia = np.argsort(-S, axis=1)[:, :k]
    ib = np.argsort(-S, axis=0)[:k, :]
    p1 = pd.DataFrame({"id_a": np.repeat(A.id.to_numpy(), k), "id_b": B.id.to_numpy()[ia.ravel()]})
    p2 = pd.DataFrame({"id_a": A.id.to_numpy()[ib.T.ravel()], "id_b": np.repeat(B.id.to_numpy(), k)})
    return pd.concat([p1, p2]).drop_duplicates().reset_index(drop=True)

K = 5
blockings = {"same model code": same_code, f"TF-IDF top {K}": top_k(S_tfidf, A, B, K),
             f"embeddings top {K}": top_k(S_embed, A, B, K)}
blockings["all three together"] = pd.concat(blockings.values()).drop_duplicates().reset_index(drop=True)
blocking_table = pd.DataFrame({k: blocking_measures(v, matches, len(A), len(B)) for k, v in blockings.items()}).T
print(blocking_table.to_string())

> **Check.** Together, the three blockings keep 1,087.0 of the 1,097 true matches (pair completeness 0.9909) with 10,205 candidates: less than 1% of all pairs.

> **Your turn.** Change one thing: use `k_small = 3` instead of 5 for both top-k blockings. Compute the union with `same_code`, and its measures.

In [ ]:
k_small = 3
small = pd.concat([same_code, top_k(S_tfidf, A, B, k_small), top_k(S_embed, A, B, k_small)]).drop_duplicates()
small_measures = blocking_measures(small, matches, len(A), len(B))
print(small_measures)

In [ ]:
expect_hash('small_measures', small_measures, '11376c64c68aabe8')

> **Check.** Fewer candidates, and a few more true matches lost. On a large catalogue, that trade decides the cost of every later step.

## 6. Compare

Each candidate pair gets a **comparison vector**: one similarity per piece of evidence. `code_in_text` says whether an Abt model code appears anywhere in the Buy title or description. A missing value (no code, no price) stays missing.

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

def numbers(title):
    return set(re.findall(r"\d+", title))

def compare(pairs, A, B, S_tfidf, S_embed):
    """The comparison vector of each candidate pair: one similarity per piece of evidence (missing = NaN)."""
    a = A.set_index("id").loc[pairs.id_a].reset_index(drop=True)
    b = B.set_index("id").loc[pairs.id_b].reset_index(drop=True)
    ra, rb = pairs.id_a.map(dict(zip(A.id, range(len(A))))).to_numpy(), pairs.id_b.map(dict(zip(B.id, range(len(B))))).to_numpy()
    v = pd.DataFrame(index=pairs.index)
    v["title_tsr"] = [fuzz.token_set_ratio(x, y) / 100 for x, y in zip(a.title, b.title)]
    v["title_jw"] = [JaroWinkler.similarity(x, y) for x, y in zip(a.title, b.title)]
    v["tfidf"] = S_tfidf[ra, rb]
    v["code_eq"] = [float(bool(x & y)) for x, y in zip(a.codes, b.codes)]
    v["code_sim"] = [max((fuzz.ratio(p, q) / 100 for p in x for q in y), default=np.nan) for x, y in zip(a.codes, b.codes)]
    v["code_in_text"] = [float(any(c in t for c in x)) if x else np.nan for x, t in zip(a.codes, b.text)]
    na, nb = a.title.map(numbers), b.title.map(numbers)
    v["numbers_jac"] = [len(x & y) / len(x | y) if x | y else np.nan for x, y in zip(na, nb)]
    v["price_ratio"] = (np.minimum(a.price, b.price) / np.maximum(a.price, b.price)).to_numpy()
    v["embed_cos"] = S_embed[ra, rb]
    return v

t0 = time.perf_counter()
cand = with_labels(blockings["all three together"], matches, split)
vectors = compare(cand, A, B, S_tfidf, S_embed)
compare_seconds = time.perf_counter() - t0
print(f"{len(cand):,} candidate pairs, {cand.same.sum():,} true; {compare_seconds:.1f} s")
print(vectors[cand.same].head(2).round(3).to_string())

## 7. Classify, and compare the methods under one measure

Every method is measured the same way: on the test split, at the threshold with the best F1 on validation. A second measure fits the team's rule: the **precision at recall 0.8**, at the threshold that found 80% of the validation matches. The first rows have no training: rules, and one similarity alone.

In [ ]:
def measures_at(score, threshold, which="test"):
    return pair_measures(cand[(score >= threshold) & (cand.split == which).to_numpy()], matches, split, which)

def choose_threshold(score, goal="f1", recall=0.8):
    """A threshold from the validation split: the best F1, or the highest threshold that reaches a recall (None if
    no threshold reaches it)."""
    best, best_t = -1, None
    for t in np.unique(np.round(score[(cand.split == "validation").to_numpy()], 4))[::-1]:
        m_ = measures_at(score, t, "validation")
        if goal == "f1" and m_["f1"] > best:
            best, best_t = m_["f1"], t
        if goal == "recall" and m_["recall"] >= recall:
            return t
    return best_t if goal == "f1" else None

results = {}
def add_result(name, score, threshold=None):
    """One row of the comparison table, on the test split: the measures at the best-F1 threshold (or a fixed one
    for a rule), and the precision at the threshold that reached recall 0.8 on validation."""
    t = choose_threshold(score) if threshold is None else threshold
    t80 = choose_threshold(score, "recall", 0.8) if threshold is None else None
    at80 = measures_at(score, t80) if t80 is not None else {"precision": None, "recall": None}
    results[name] = {**measures_at(score, t), "threshold": round(float(t), 3),
                     "precision at recall 0.8": at80["precision"], "recall there": at80["recall"]}
    return results[name]

title_a, title_b = cand.id_a.map(A.set_index("id").title), cand.id_b.map(B.set_index("id").title)
add_result("exact title (normalised)", title_a.eq(title_b).to_numpy().astype(float), threshold=1.0)
add_result("rule: same model code", vectors.code_eq.to_numpy(), threshold=1.0)
add_result("TF-IDF alone", vectors.tfidf.to_numpy())
add_result("embeddings alone", vectors.embed_cos.to_numpy())
print(pd.DataFrame(results).T.to_string())

Splink learns its weights **without labels**, as in Module 5. It compares the title words, the first word (the brand) and the model codes.

In [ ]:
import logging, duckdb, splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on
from splink.backends.duckdb import DuckDBAPI
logging.getLogger("splink").setLevel(logging.WARNING)
Path("duckdb_tmp").mkdir(exist_ok=True)
con = duckdb.connect()   # DuckDB may write temporary files: keep them in one folder, at most 4 GB
con.execute("SET threads = 1; SET memory_limit = '2GB'; SET temp_directory = 'duckdb_tmp'; SET max_temp_directory_size = '4GB'")

def splink_table(P):
    return pd.DataFrame({"unique_id": P.id, "brand": P.title.str.split().str[0], "codes": P.codes.map(sorted),
                         "words": P.title.str.split().map(lambda w: sorted(set(w)))})

settings = SettingsCreator(
    link_type="link_only", unique_id_column_name="unique_id",
    comparisons=[cl.ArrayIntersectAtSizes("words", [5, 4, 3, 2]), cl.ExactMatch("brand"), cl.ArrayIntersectAtSizes("codes", [1])],
    blocking_rules_to_generate_predictions=[block_on("brand"), "list_has_any(l.codes, r.codes)"])
t0 = time.perf_counter()
api = DuckDBAPI(connection=con)
linker = Linker([api.register(splink_table(A)), api.register(splink_table(B))], settings)
for name in ("splink", "splink.internals"):   # Splink prints every training step: show only warnings
    logging.getLogger(name).setLevel(logging.WARNING)
linker.training.estimate_probability_two_random_records_match(["list_has_any(l.codes, r.codes)"], recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=1e8, seed=1)   # all 1.2 million pairs
linker.training.estimate_parameters_using_expectation_maximisation(block_on("brand"))
linker.training.estimate_parameters_using_expectation_maximisation("list_has_any(l.codes, r.codes)")
splink_seconds = time.perf_counter() - t0
scored = linker.inference.predict().as_pandas_dataframe()
weight = dict(zip(zip(scored.unique_id_l, scored.unique_id_r), scored.match_weight))
splink_score = np.array([weight.get((x, y), -99.0) for x, y in zip(cand.id_a, cand.id_b)])
print(f"trained without labels in {splink_seconds:.1f} s;", add_result("Splink, no labels", splink_score))

> **Predict.** About two thirds of the true candidate pairs share a model code. What m probability ("the share of true matches that agree on this field") do you expect Splink to learn for the codes?

In [ ]:
levels = [(c["output_column_name"], l["label_for_charts"], l["m_probability"], l["u_probability"])
          for c in linker.misc.save_model_to_json()["comparisons"] for l in c["comparison_levels"] if not l.get("is_null_level")]
codes_m = [round(mp, 3) for col, lab, mp, up in levels if col == "codes" and ">=" in lab][0]
share_codes = round(float(vectors.code_eq[cand.same].mean()), 3)
print("Splink's m for 'share a model code':", codes_m, "| share of true candidate pairs that share a code:", share_codes)

> **Check.** Splink learned a very low m for the codes. Without labels, expectation maximisation found a different group of pairs: products of one brand and one product line, which share words but rarely a code. So its codes evidence is much too weak. The labels fix that.

Now the classifiers that learn from the labelled training pairs:

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = (cand.split == "train").to_numpy()
FIELDS = [c for c in vectors.columns if c != "embed_cos"]
def flat(x):   # logistic regression needs numbers: missing -> 0, plus a "missing" column
    return pd.concat([x.fillna(0), x.isna().astype(float).add_suffix("_missing")], axis=1)

t0 = time.perf_counter()
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(flat(vectors[FIELDS])[train], cand.same[train])
lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20, random_state=1,
                      n_jobs=1, verbose=-1).fit(vectors[FIELDS][train], cand.same[train])
lgbm_score = lgbm.predict_proba(vectors[FIELDS])[:, 1]
learn_seconds = time.perf_counter() - t0
add_result("logistic regression", logreg.predict_proba(flat(vectors[FIELDS]))[:, 1])
add_result("LightGBM", lgbm_score)
lgbm_embed = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20, random_state=1,
                            n_jobs=1, verbose=-1).fit(vectors[train], cand.same[train])
add_result("LightGBM + embedding cosine", lgbm_embed.predict_proba(vectors)[:, 1])
print(f"trained on {train.sum():,} labelled pairs ({cand.same[train].sum()} true) in {learn_seconds:.1f} s")
print(pd.DataFrame(results).T[["precision", "recall", "f1", "precision at recall 0.8"]].to_string())

> **Check.** LightGBM: F1 0.869 (precision 0.93, recall 0.816). Logistic regression is close. The embedding cosine alone is the weakest score, and adding it to LightGBM changes little.

Which evidence does LightGBM use?

In [ ]:
gain = pd.Series(lgbm.booster_.feature_importance("gain"), index=FIELDS)
print((gain / gain.sum()).round(3).sort_values(ascending=False).to_string())

> **Check.** More than half of the gain comes from `code_in_text`: the model number found in the other shop's text. The title similarities come next.

## 8. One product, one partner

Each product in one shop is the same as **at most one** product in the other shop (almost always). So when one Abt product has two good candidates, only the best one can be right. **One-to-one matching** keeps a pair only when it is the best pair of both its products. A tiny example first:

In [ ]:
tiny = pd.DataFrame({"id_a": ["A1", "A1", "A2", "A2"], "id_b": ["B1", "B2", "B1", "B2"], "score": [0.90, 0.60, 0.70, 0.40]})
def one_to_one(pairs, score):
    """Keep a pair only when it has the best score of both its products (the other pairs get score 0)."""
    d = pairs.assign(s=score)
    keep = (d.s == d.groupby("id_a").s.transform("max")) & (d.s == d.groupby("id_b").s.transform("max"))
    return np.where(keep, score, 0.0)
print(tiny.assign(kept=one_to_one(tiny, tiny.score)).to_string(index=False))

> **Check.** A1–B1 (0.90) is the best pair of A1 and of B1: kept. A2's best pair is A2–B1 (0.70), but B1 has a better pair: dropped. A2–B2 is not the best pair of A2: dropped. So A2 has no partner. A wrong pair that scores high can take a partner away from the right pair.

Now on the real candidates:

In [ ]:
lgbm_11 = one_to_one(cand, lgbm_score)
print("kept by one-to-one:", int((lgbm_11 > 0).sum()), "of", len(cand))
print(add_result("LightGBM, one-to-one", lgbm_11))

> **Check.** One-to-one keeps about one candidate in ten. The F1 rises from 0.869 to 0.924, and the precision from 0.93 to 0.976.

> **Your turn.** Apply `one_to_one()` to the logistic regression's probability, and add it to the table.

In [ ]:
logreg_11 = one_to_one(cand, logreg.predict_proba(flat(vectors[FIELDS]))[:, 1])
logreg_11_f1 = add_result("logistic regression, one-to-one", logreg_11)["f1"]
print(logreg_11_f1)

In [ ]:
expect('logreg_11_f1', logreg_11_f1, 0.915, tolerance=0.001)

## 9. Decide: accept, review or no match

The team accepts a match automatically only when at least 99 in 100 such matches are right (our assumption for this case study). Pairs between a lower and an upper threshold go to a person. The table is on the validation split:

In [ ]:
def band_table(score, lower, upper, which="validation"):
    """Three bands: accept automatically (score >= upper), review (lower <= score < upper), reject the rest."""
    rows = (cand.split == which).to_numpy()
    s = cand[rows].assign(score=score[rows])
    acc, rev = s[s.score >= upper], s[(s.score >= lower) & (s.score < upper)]
    true_n = sum(split["a" + x] == which for x in matches.id_a)
    return {"accepted": len(acc), "accepted right": int(acc.same.sum()), "precision accepted": round(float(acc.same.mean()), 3),
            "to review": len(rev), "true in review": int(rev.same.sum()),
            "recall if reviews are right": round(float((acc.same.sum() + rev.same.sum()) / true_n), 3)}

grid = pd.DataFrame({(lo, up): band_table(lgbm_11, lo, up) for lo in (0.01, 0.05, 0.2) for up in (0.5, 0.8, 0.95)}).T
grid.index.names = ["lower", "upper"]
print(grid.to_string())

In [ ]:
TARGET = 0.99                # the team's rule: at least 99 in 100 automatic matches right (an assumption)
UPPER = min(up for (lo, up), row in grid.iterrows() if row["precision accepted"] >= TARGET)
LOWER = 0.01                 # 0.01 instead of 0.05: 5 more reviews find 4 more true matches (see the table)
decision = {w: band_table(lgbm_11, LOWER, UPPER, w) for w in ("validation", "test")}
print("upper", UPPER, "lower", LOWER)
print(pd.DataFrame(decision).T.to_string())

> **Check.** Upper threshold 0.8, lower 0.01. On test, 218 pairs are accepted (216 right) and 31 go to review.

Where do the other true matches of the test split go? The next cell counts them, step by step.

In [ ]:
# Where do the test split's true matches go?
test_true = matches[[split["a" + x] == "test" for x in matches.id_a]]
c = cand.assign(score=lgbm_11, raw=lgbm_score)
c = c[c.split == "test"]
key = lambda d: set(zip(d.id_a, d.id_b))
in_cand, kept = key(test_true) & key(c), key(c[c.score > 0])
true_kept = {p for p in kept if p in key(test_true)}
partner_taken = {p for p in in_cand - kept if p[0] in {a for a, _ in true_kept} or p[1] in {b for _, b in true_kept}}
losses = {"true matches": len(test_true), "lost by blocking": len(key(test_true) - in_cand),
          "dropped: the product's other true partner won": len(partner_taken),
          "dropped: a wrong pair won": len(in_cand - kept - partner_taken),
          "kept, but below the lower threshold": len({p for p in true_kept if c.set_index(["id_a", "id_b"]).score[p] < LOWER}),
          "accepted or reviewed": len({p for p in true_kept if c.set_index(["id_a", "id_b"]).score[p] >= LOWER})}
print(losses)

> **Check.** The blocking loses only 1. Most losses come from one-to-one: 7 because the product's other true partner won, and 23 because a wrong pair won.

### A language model for the review band (recorded)

We asked a small chat model (Azure chat-small) about every unsure pair of the validation and test splits, once, and recorded its answers. You need no key: the notebook reads the recorded file.

In [ ]:
recorded = pd.read_json("data/recorded/llm_band.jsonl", lines=True, dtype={"id_a": str, "id_b": str})
answer = dict(zip(zip(recorded.id_a, recorded.id_b), recorded.answer))
in_band = (lgbm_11 >= LOWER) & (lgbm_11 < UPPER) & (cand.split != "train").to_numpy()
band = cand[in_band].assign(answer=[answer.get(p, "not recorded") for p in zip(cand.id_a[in_band], cand.id_b[in_band])])
print(pd.crosstab(band.answer, band.same.map({True: "true match", False: "not a match"})))
said_same = np.array([answer.get(p) == "same" for p in zip(cand.id_a, cand.id_b)])
llm_called = cand[(lgbm_11 >= UPPER) | (in_band & said_same)]
results["LightGBM, one-to-one, LLM decides the band"] = {**pair_measures(llm_called, matches, split), "threshold": UPPER,
                                                          "precision at recall 0.8": None, "recall there": None}
print(results["LightGBM, one-to-one, LLM decides the band"])

> **Check.** Read the table: how often does the model say "same" for a pair that is not a match? Those are false matches that nobody checks. The recorded answers are from one model on one day; your own run would differ.

What does each step cost? The prices are assumptions (the course's price table): a CPU server at US$0.17 an hour, and the chat model at US$0.10 and US$0.50 per million input and output tokens.

In [ ]:
costs = {"CPU server, US$ an hour": 0.17, "LLM US$ per 1M tokens in": 0.10, "LLM US$ per 1M tokens out": 0.50,
         "review minutes a pair": 1.2, "reviewer US$ an hour": 40}   # assumptions: the course's price table
cpu_seconds = compare_seconds + learn_seconds
per_1000_cpu = cpu_seconds / len(cand) * 1000 / 3600 * costs["CPU server, US$ an hour"]
asked = recorded.merge(band[["id_a", "id_b"]], on=["id_a", "id_b"])
per_1000_llm = (asked.tokens_in.mean() * costs["LLM US$ per 1M tokens in"] + asked.tokens_out.mean() * costs["LLM US$ per 1M tokens out"]) / 1e6 * 1000
band_share = in_band.sum() / (cand.split != "train").sum()
test_products = sum(split["a" + x] == "test" for x in abt.id)
review_per_1000 = decision["test"]["to review"] / test_products * 1000
print(f"CPU: {cpu_seconds:.1f} s for {len(cand):,} pairs -> US${per_1000_cpu:.7f} per 1,000 candidate pairs")
print(f"LLM: {asked.tokens_in.mean():.0f} tokens in, {asked.tokens_out.mean():.0f} out a pair -> US${per_1000_llm:.3f} per 1,000 pairs it reads;"
      f" median {asked.seconds.median():.2f} s a pair; {band_share:.1%} of candidate pairs are in the band")
print(f"Review: {review_per_1000:.0f} pairs per 1,000 Abt products -> {review_per_1000 * costs['review minutes a pair'] / 60:.1f} hours,"
      f" US${review_per_1000 * costs['review minutes a pair'] / 60 * costs['reviewer US$ an hour']:.0f}")

## 10. The whole design on new input

`match_shops()` runs every step of the chosen design. You run it on the Abt products of the test split, which no step has used, against all of Buy.

In [ ]:
def match_shops(A, B, S_tfidf, S_embed, model, lower=LOWER, upper=UPPER, k=K):
    """The whole design on two prepared shops: block, compare, score, one-to-one, three bands."""
    pairs = pd.concat([code_pairs(A, B), top_k(S_tfidf, A, B, k), top_k(S_embed, A, B, k)]).drop_duplicates().reset_index(drop=True)
    score = one_to_one(pairs, model.predict_proba(compare(pairs, A, B, S_tfidf, S_embed)[FIELDS])[:, 1])
    out = pairs.assign(score=score.round(3), decision=np.select([score >= upper, score >= lower], ["accept", "review"], "no match"))
    return out[out.decision != "no match"].sort_values("score", ascending=False).reset_index(drop=True)

In [ ]:
# New input: the test products of Abt, which no step has used, matched against all of Buy.
new_abt = abt[abt.id.map(lambda x: split["a" + x] == "test")]
A_new = prepare(new_abt)
S_t = S_tfidf[A.id.isin(new_abt.id).to_numpy()]; S_e = S_embed[A.id.isin(new_abt.id).to_numpy()]
t0 = time.perf_counter()
run = match_shops(A_new, B, S_t, S_e, lgbm)
run_seconds = time.perf_counter() - t0
run = with_labels(run, matches, split)
print(f"{len(new_abt)} new products, {run_seconds:.1f} s:", run.decision.value_counts().to_dict())
print(run.groupby("decision").same.agg(["count", "sum"]).rename(columns={"sum": "true"}).to_string())

The pairs that go to a person:

In [ ]:
show = run.assign(Abt=run.id_a.map(names_a), Buy=run.id_b.map(names_b))
print(show[show.decision == "review"][["score", "same", "Abt", "Buy"]].head(6).to_string(index=False))

> **Check.** Most review pairs are true matches with weak evidence (no shared code, short titles). The pairs that are not matches are often an accessory or a close model of the same line.

## 11. Your turn: the harder pair of shops

The same page has a harder benchmark: Amazon and Google's shopping service, mostly software. The setup cell downloaded it too.

In [ ]:
amazon, google, ag_matches = load("amazon-googleproducts", "Amazon.csv", "GoogleProducts.csv",
                                  "Amzon_GoogleProducts_perfectMapping.csv")
print("Amazon", amazon.shape, "| Google", google.shape, "| true matches", len(ag_matches))
print("products with a match:", f"Amazon {amazon.id.isin(ag_matches.id_a).sum()} of {len(amazon)},",
      f"Google {google.id.isin(ag_matches.id_b).sum()} of {len(google)}")
print(pd.DataFrame({"Amazon": ag_matches.id_a.map(amazon.set_index("id").name).head(4).to_numpy(),
                    "Google": ag_matches.id_b.map(google.set_index("id").name).head(4).to_numpy()}).to_string(index=False))

> **Your turn.** Run the whole case on Amazon and Google: call `whole_case()` with the two tables, the matches, and the names `"amazon"` and `"google"` (for the shipped embeddings). It takes about a minute.

In [ ]:
def whole_case(a, b, m, shop_a, shop_b):
    """The chosen design on another pair of shops: split, prepare, block, compare, train, choose, measure."""
    sp = split_products(a, b, m)
    PA, PB = prepare(a), prepare(b)
    St, Se = similarity_tables(PA, PB, shop_a, shop_b)
    pairs = pd.concat([code_pairs(PA, PB), top_k(St, PA, PB, K), top_k(Se, PA, PB, K)]).drop_duplicates().reset_index(drop=True)
    c = with_labels(pairs, m, sp)
    v = compare(c, PA, PB, St, Se)[FIELDS]
    tr = (c.split == "train").to_numpy()
    model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20, random_state=1,
                           n_jobs=1, verbose=-1).fit(v[tr], c.same[tr])
    s = one_to_one(c, model.predict_proba(v)[:, 1])
    best = max((pair_measures(c[(s >= t) & (c.split == "validation").to_numpy()], m, sp, "validation")["f1"], t)
               for t in (0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9))
    t = best[1]
    return {"blocking": blocking_measures(pairs, m, len(PA), len(PB)), "threshold": t,
            "test": pair_measures(c[(s >= t) & (c.split == "test").to_numpy()], m, sp)}

ag_result = whole_case(amazon, google, ag_matches, "amazon", "google")
print(ag_result)
ag_f1 = ag_result['test']['f1']

In [ ]:
expect('ag_f1', ag_f1, 0.633, tolerance=0.001)

> **Check.** The same design gets a much lower F1. Software titles have few model numbers, and many products differ only by an edition or a version ("2007" against "2008", "upgrade" against "full"). Not every product has a match, so one-to-one can also keep a wrong partner. Read the lesson page for what to change.

## 12. Recap

- Exact names found almost nothing. Normalised model codes are precise, but only for two thirds of the matches.
- Three blockings together kept 99% of the true matches with less than 1% of the pairs.
- LightGBM on the comparison vector beat the rules, Splink without labels and the embeddings.
- One-to-one matching was the biggest single gain on this benchmark, because every product has a partner.
- The decision: accept above the upper threshold, review the band, and say what the review costs.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Link bibliographic records](https://learning.nextia-ai.com/courses/record-matching/m09/link-bibliographic-records/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m09/match-products-between-two-shops/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The product offers are real listings from two shops, published by the University of Leipzig under CC BY 4.0.